# Next-Word Prediction with a Keras Self-Attention Block

Instead of an LSTM, this model uses one Transformer-style block (MultiHeadAttention + feed-forward + LayerNormalization) built with the Keras functional API. It is trained on the opening of *Around the World in Eighty Days* with a train/validation split and predicts the next word of a sentence.

**How to run:** Open in Google Colab, choose **Runtime > Change runtime type > GPU**, then **Runtime > Run all**.

_Rewritten from my original Colab notebook: `friday predect text code.ipynb` and `Copy of friday predect text code.ipynb`._

## 1. Imports

In [ ]:
import re
import urllib.request

import numpy as np
import tensorflow as tf
import keras
from keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)  # seeds Python, NumPy and TensorFlow
print("TensorFlow:", tf.__version__)

from sklearn.model_selection import train_test_split

## 2. Download the text

In [ ]:
def download_gutenberg_book(url):
    """Download a plain-text book from Project Gutenberg and remove the licence header/footer."""
    raw = urllib.request.urlopen(url).read().decode("utf-8-sig")
    start = raw.find("*** START OF")
    end = raw.find("*** END OF")
    if start != -1 and end != -1:
        raw = raw[raw.find("\n", start) + 1:end]
    return raw.replace("\r\n", "\n").strip()

In [ ]:
book = download_gutenberg_book("https://www.gutenberg.org/cache/epub/103/pg103.txt")
start = book.find("Mr. Phileas Fogg lived")
text = " ".join(book[start:start + 40000].split())
sentences = [s for s in re.split(r"(?<=[.!?;])\s+", text) if len(s.split()) > 2]
print("Sentences:", len(sentences))

## 3. Vocabulary and samples

In [ ]:
def clean_words(line):
    """Lower-case a line and keep only words (letters and apostrophes)."""
    line = line.lower().replace("\u2019", "'")
    return re.findall(r"[a-z']+", line)


def build_vocab(lines):
    """Map every word to an integer id. Id 0 is reserved for padding."""
    words = sorted({w for line in lines for w in clean_words(line)})
    word_to_id = {w: i + 1 for i, w in enumerate(words)}
    id_to_word = {i: w for w, i in word_to_id.items()}
    return word_to_id, id_to_word


def encode(line, word_to_id):
    return [word_to_id[w] for w in clean_words(line) if w in word_to_id]

In [ ]:
def make_ngram_dataset(lines, word_to_id, max_len=None):
    """For each line 'a b c d' create the samples [a]->b, [a b]->c, [a b c]->d."""
    sequences = []
    for line in lines:
        ids = encode(line, word_to_id)
        for i in range(1, len(ids)):
            sequences.append(ids[: i + 1])
    if max_len is None:
        max_len = max(len(s) for s in sequences)
    padded = keras.utils.pad_sequences(sequences, maxlen=max_len, padding="pre", truncating="pre")
    X, y = padded[:, :-1], padded[:, -1]
    return X, y, max_len

In [ ]:
word_to_id, id_to_word = build_vocab(sentences)
vocab_size = len(word_to_id) + 1
X, y, max_len = make_ngram_dataset(sentences, word_to_id, max_len=30)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Vocabulary:", vocab_size)

## 4. Build the attention model
A position embedding is added so the model knows word order (attention alone does not). A small custom layer adds the two embeddings together.

In [ ]:
EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT = 128, 2, 128, 0.1
seq_len = max_len - 1


class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, seq_len, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = layers.Embedding(seq_len, embed_dim)

    def call(self, x):
        positions = keras.ops.arange(keras.ops.shape(x)[-1])
        return self.token_emb(x) + self.pos_emb(positions)


inputs = keras.Input(shape=(seq_len,))
x = TokenAndPositionEmbedding(seq_len, vocab_size, EMBED_DIM)(inputs)

attention = layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=EMBED_DIM // NUM_HEADS)(x, x)
x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(DROPOUT)(attention))

ffn = layers.Dense(FF_DIM, activation="relu")(x)
ffn = layers.Dense(EMBED_DIM)(ffn)
x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(DROPOUT)(ffn))

x = layers.GlobalMaxPooling1D()(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.Adam(1e-3), metrics=["accuracy"])
model.summary()

## 5. Train

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, batch_size=64, callbacks=[early_stop])

## 6. Evaluate

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="train loss")
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch")
plt.legend()
plt.show()

In [ ]:
loss, acc = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation loss: {loss:.3f} | accuracy: {acc:.3f}")

## 7. Predict

In [ ]:
def predict_next_word(model, seed_text, word_to_id, id_to_word, max_len):
    ids = encode(seed_text, word_to_id)
    padded = keras.utils.pad_sequences([ids], maxlen=max_len - 1, padding="pre", truncating="pre")
    probs = model.predict(padded, verbose=0)[0]
    probs[0] = 0  # never predict the padding id
    return id_to_word.get(int(np.argmax(probs)), "")


def generate_text(model, seed_text, n_words, word_to_id, id_to_word, max_len):
    text = seed_text
    for _ in range(n_words):
        next_word = predict_next_word(model, text, word_to_id, id_to_word, max_len)
        if not next_word:
            break
        text += " " + next_word
    return text

In [ ]:
print(generate_text(model, "Mr Phileas Fogg was", 10, word_to_id, id_to_word, max_len))
print(generate_text(model, "He was never seen", 10, word_to_id, id_to_word, max_len))

Because of the custom layer, we save only the weights (simplest for a beginner project).

In [ ]:
model.save_weights("fogg_attention.weights.h5")